# 04 - Image Segmentation (Week 4)
Training a UNet for periapical lesion segmentation using box-derived pseudo-masks.


In [ ]:
import sys, os, json
# Find project root (works from project root or notebooks/ subdir)
_root = None
for _p in [os.path.abspath('.'), os.path.abspath('..')]:
    if os.path.exists(os.path.join(_p, 'src', 'config.py')):
        _root = _p
        break
if _root:
    sys.path.append(_root)
else:
    raise RuntimeError('Could not find project root (src/config.py not found)')
from src.config import *
from src.train_segmentation import train_segmentation, evaluate_segmentation
from src.visualize import plot_training_history
import warnings; warnings.filterwarnings('ignore')


In [ ]:
# Load segmentation manifest
seg_dir = os.path.join(DATA_PROCESSED, 'segmentation')
import pickle
with open(os.path.join(DATA_PROCESSED, 'splits.pkl'), 'rb') as f:
    split_info = pickle.load(f)

def load_seg_records(split_name):
    img_dir = os.path.join(seg_dir, 'images', split_name)
    mask_dir = os.path.join(seg_dir, 'masks', split_name)
    records = []
    for fname in os.listdir(img_dir):
        base = os.path.splitext(fname)[0]
        mask_path = os.path.join(mask_dir, base + '.png')
        if os.path.exists(mask_path):
            records.append({
                'image_path': os.path.join(img_dir, fname),
                'mask_path': mask_path,
                'filename': fname,
            })
    return records

train_records = load_seg_records('train')
val_records = load_seg_records('val')
test_records = load_seg_records('test')
print(f'Train: {len(train_records)}, Val: {len(val_records)}, Test: {len(test_records)}')


In [ ]:
# Train segmentation model
model, history = train_segmentation(train_records, val_records)


In [ ]:
# Plot training history
plot_training_history(history, 'Segmentation',
    os.path.join(OUTPUTS_DIR, 'segmentation', 'training_history.png'))
print('Training history plot saved.')


In [ ]:
# Evaluate on test set
metrics, pred_masks, true_masks = evaluate_segmentation(model, test_records)


In [ ]:
# Visualize a sample segmentation
from src.visualize import visualize_segmentation
if test_records:
    rec = test_records[0]
    out_path = os.path.join(OUTPUTS_DIR, 'segmentation', 'sample_segmentation.png')
    visualize_segmentation(rec['image_path'], pred_masks[0].cpu(), out_path)
    print(f'Saved: {out_path}')


In [ ]:
# Save metrics
with open(os.path.join(OUTPUTS_DIR, 'segmentation', 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print('Segmentation complete!')
